# ISIC 2017 — Download Data & Train Classifiers

This notebook downloads the full ISIC 2017 dataset, trains ResNet-18 and
SqueezeNet 1.1, and saves the trained weights to Google Drive.

**Prerequisites**: Set the runtime to **GPU** (Runtime → Change runtime type → T4 GPU).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Where outputs will be persisted on Drive
DRIVE_ROOT = '/content/drive/MyDrive/thesis'
!mkdir -p {DRIVE_ROOT}/weights {DRIVE_ROOT}/results

## 2. Clone the repository

In [ ]:
import os
REPO_DIR = '/content/fae-metrics-master-thesis'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/dawkopagh/fae-metrics-master-thesis.git {REPO_DIR}
    %cd {REPO_DIR}

# All paths below are relative to REPO_DIR
print(f'Working directory: {os.getcwd()}')

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Download ISIC 2017 dataset

Downloads ~6 GB of images and masks from the ISIC challenge S3 bucket.
Uses `skip_existing=True` so it's safe to re-run after interruption.

In [ ]:
import logging, sys
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)])

sys.path.insert(0, '.')
from src.data.download_isic import download_isic2017

DATA_ROOT = 'data/isic2017'
download_isic2017(dest_dir=DATA_ROOT, skip_existing=True)

In [ ]:
# Verify counts
for split in ['train', 'validation', 'test']:
    for cls in ['melanoma', 'nevus', 'seborrheic_keratosis']:
        path = f'{DATA_ROOT}/images/{split}/{cls}'
        n = len([f for f in os.listdir(path) if not f.startswith('.')]) if os.path.exists(path) else 0
        print(f'{split:12s} {cls:25s} {n:5d}')

## 5. Train classifiers

Trains both models with class-weighted cross-entropy and saves the best
validation-accuracy checkpoint.

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 25
SEED = 42

In [ ]:
from src.models.train import train_classifier

# --- ResNet-18 ---
resnet_log = train_classifier(
    arch='resnet18',
    data_root=DATA_ROOT,
    output_weights_path='weights/resnet18_isic2017.pth',
    device=DEVICE,
    epochs=EPOCHS,
    seed=SEED,
)
resnet_log.to_csv('results/training_log_resnet18.csv', index=False)
print(f"\nResNet-18 best val acc: {resnet_log[resnet_log['phase']=='validation']['acc'].max():.4f}")

In [ ]:
# --- SqueezeNet 1.1 ---
sq_log = train_classifier(
    arch='squeezenet',
    data_root=DATA_ROOT,
    output_weights_path='weights/squeezenet_isic2017.pth',
    device=DEVICE,
    epochs=EPOCHS,
    seed=SEED,
)
sq_log.to_csv('results/training_log_squeezenet.csv', index=False)
print(f"\nSqueezeNet best val acc: {sq_log[sq_log['phase']=='validation']['acc'].max():.4f}")

## 6. Copy outputs to Google Drive

In [ ]:
import shutil

# Copy weights
for f in ['weights/resnet18_isic2017.pth', 'weights/squeezenet_isic2017.pth']:
    dest = f'{DRIVE_ROOT}/{f}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(f, dest)
    print(f'Copied {f} → {dest}')

# Copy training logs
for f in ['results/training_log_resnet18.csv', 'results/training_log_squeezenet.csv']:
    dest = f'{DRIVE_ROOT}/{f}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(f, dest)
    print(f'Copied {f} → {dest}')

print('\nAll outputs saved to Google Drive.')

## 7. (Optional) Run vertical slice with new weights

Quick sanity check — run the 3-metric evaluation on the test split.

In [ ]:
from src.pipeline import run_vertical_slice

df = run_vertical_slice(
    data_root=DATA_ROOT,
    resnet_weights='weights/resnet18_isic2017.pth',
    squeezenet_weights='weights/squeezenet_isic2017.pth',
    split='test',
    device=DEVICE,
    output_csv='results/vertical_slice_full.csv',
    seed=SEED,
)

print(f'\nShape: {df.shape}')
summary = df.groupby(['model', 'fae_method', 'metric'])['score'].agg(['mean', 'std', lambda x: x.isna().sum()])
summary.columns = ['mean', 'std', 'nan_count']
print(summary.to_string())

# Copy to Drive
shutil.copy2('results/vertical_slice_full.csv', f'{DRIVE_ROOT}/results/vertical_slice_full.csv')
print(f'\nSaved to Drive.')